In [1]:
from pathlib import Path
import sys
import random

import numpy as np
import torch
import yaml

PROJECT_ROOT = Path("/vol/bitbucket/ma5923/_projects/provably-safe-policy-updates")
RL_DIR = PROJECT_ROOT / "rl_project"
EXP_DIR = RL_DIR / "experiments"
FROZEN_DIR = EXP_DIR / "frozen_lake"

for p in (PROJECT_ROOT, RL_DIR, EXP_DIR, FROZEN_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from rl_project.experiments.frozen_lake.frozenlake_utils import make_frozenlake_env
from rl_project.utils.ppo_utils import PPOConfig, ppo_train


N_ACTIONS = 4


def set_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_actor(obs_dim: int, hidden: int = 64) -> torch.nn.Sequential:
    return torch.nn.Sequential(
        torch.nn.Linear(obs_dim, hidden),
        torch.nn.ReLU(),
        torch.nn.Linear(hidden, hidden),
        torch.nn.ReLU(),
        torch.nn.Linear(hidden, N_ACTIONS),
    )


def make_critic(obs_dim: int, hidden: int = 64) -> torch.nn.Sequential:
    return torch.nn.Sequential(
        torch.nn.Linear(obs_dim, hidden),
        torch.nn.ReLU(),
        torch.nn.Linear(hidden, hidden),
        torch.nn.ReLU(),
        torch.nn.Linear(hidden, 1),
    )


def train_frozenlake_ppo(
    cfg_name: str = "standard_4x4",
    seed: int = 0,
    total_steps: int | None = None,
    hidden: int | None = None,
):
    set_seeds(seed)

    with open(FROZEN_DIR / "configs.yaml") as f:
        all_cfgs = yaml.safe_load(f)

    cfg_yaml = all_cfgs[cfg_name]
    train_cfg = cfg_yaml.get("train", {})

    env_map = cfg_yaml["env1_map"]
    is_slippery = bool(cfg_yaml.get("is_slippery", False))

    total_steps = total_steps or train_cfg.get("source_total_timesteps", 500_000)
    hidden = hidden or train_cfg.get("hidden", 64)

    env = make_frozenlake_env(
        env_map=env_map,
        task_num=0,
        is_slippery=is_slippery,
    )

    obs_dim = env.observation_space.shape[0]
    actor = make_actor(obs_dim, hidden)
    critic = make_critic(obs_dim, hidden)

    ppo_cfg = PPOConfig(
        seed=seed,
        total_timesteps=total_steps,
        eval_episodes=1 if not is_slippery else 100,
        rollout_steps=256,
        update_epochs=8,
        minibatch_size=64,
        gamma=0.99,
        gae_lambda=0.95,
        clip_coef=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        lr=3e-4,
        max_grad_norm=0.5,
        early_stop=not is_slippery,
        early_stop_min_steps=0,
        early_stop_deterministic_total_reward_threshold=1.0 if not is_slippery else None,
        early_stop_deterministic_eval_episodes=1,
        device="cpu",
    )

    actor, critic, training_data = ppo_train(
        env=env,
        cfg=ppo_cfg,
        actor_warm_start=actor,
        critic_warm_start=critic,
        return_training_data=True,
    )

    env.close()
    return actor.cpu(), critic.cpu(), training_data, cfg_yaml


actor, critic, training_data, frozen_cfg = train_frozenlake_ppo(
    cfg_name="standard_4x4",
    seed=0,
)

Use PGD: False
Steps=2560 | meanR=0.0 +/- 0.0 | elapsed=9.9s | failure_rate=1.00 | det_meanR=0.00 (1 ep)
Steps=5120 | meanR=1.0 +/- 0.0 | elapsed=12.1s | failure_rate=0.00 | det_meanR=1.00 (1 ep)
  [Early stop] step=5120 | meanR=1.00 (threshold=None) | failure_rate=0.00 (threshold=None) | det_meanR=1.00 (threshold=1.0, episodes=1)
Final evaluation over 1 episodes: mean_reward=1.00 +/- 0.00 | failure_rate=0.00


## Shield synthesis

In [23]:
from utils.shield_utils import synthesise_shield
import types

def frozenlake_transition_matrix(env):
    """
    MASA expects shape: (n_states, n_states, n_actions),
    with P[s_next, s, a] = Pr(s_next | s, a).
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    T = np.zeros((n_states, n_states, n_actions), dtype=np.float64)

    for s in range(n_states):
        for a in range(n_actions):
            for prob, next_s, reward, terminated in env.P[s][a]:
                T[next_s, s, a] += prob

    return T

def frozenlake_label_fn(obs):
    """
    Labels states as unsafe if the cell is a hole.
    FrozenLake desc contains: S, F, H, G.
    """
    s = int(obs)
    row, col = divmod(s, shield_env.unwrapped.ncol)
    cell = shield_env.unwrapped.desc[row, col].decode("utf-8")

    if cell == "H":
        return {"unsafe", "hole"}
    if cell == "G":
        return {"goal"}
    return set()


def cost_fn(labels):
    return 1.0 if "unsafe" in labels else 0.0

safe_state_action_pairs = synthesise_shield(
    env=shield_env,
    transition_matrix_fn=lambda env: env.get_transition_matrix(),
    label_fn=label_fn,
    cost_fn=cost_fn,
)

Calculating the maximum number of successor states ...
Calculated maximum number of successor states [4] ...
Building successor state matrix and probabilities ...
Computing almost sure safe set ...


In [24]:
cfg_name: str = "standard_4x4"

with open(FROZEN_DIR / "configs.yaml") as f:
    all_cfgs = yaml.safe_load(f)

cfg_yaml = all_cfgs[cfg_name]
train_cfg = cfg_yaml.get("train", {})

env_map = cfg_yaml["env1_map"]
is_slippery = bool(cfg_yaml.get("is_slippery", False))

shield_env = make_frozenlake_env(
    env_map=env_map,
    task_num=0,
    is_slippery=is_slippery,
)
# Add the interface expected by MASA's prob_shield helper.
shield_env.unwrapped.has_transition_matrix = True
shield_env.unwrapped.has_successor_states_dict = False
shield_env.unwrapped.get_transition_matrix = types.MethodType(
    lambda self: frozenlake_transition_matrix(self),
    shield_env.unwrapped,
)
# Optional but useful: MASA checks initial state feasibility if _start_state exists.
shield_env.unwrapped._start_state = 0

# start_state = int(shield_env._start_state)
# safe_actions_at_start = np.flatnonzero(safe_state_action_pairs[start_state]).tolist()
# states_with_safe_actions = int((safe_state_action_pairs.sum(axis=1) > 0).sum())
# total_safe_pairs = int(safe_state_action_pairs.sum())

# print(f"Shield shape: {safe_state_action_pairs.shape}")
# print(f"States with at least one safe action: {states_with_safe_actions}")
# print(f"Total safe state-action pairs: {total_safe_pairs}")
# print(f"Safe actions at actual start state {start_state}: {safe_actions_at_start}")

# if not safe_actions_at_start:
#     print("The actual start state is outside the almost-sure winning set under the slip dynamics.")

shield_env.close()

In [25]:
safe_state_action_pairs = synthesise_shield(
    shield_env, frozenlake_transition_matrix, frozenlake_label_fn, cost_fn)
print(safe_state_action_pairs)

Calculating the maximum number of successor states ...
Calculated maximum number of successor states [4] ...
Building successor state matrix and probabilities ...
Computing almost sure safe set ...
[[1 1 1 1]
 [1 0 1 1]
 [1 1 1 1]
 [1 0 1 1]
 [1 1 0 1]
 [0 0 0 0]
 [0 1 0 1]
 [0 0 0 0]
 [1 0 1 1]
 [1 1 1 0]
 [1 1 0 1]
 [0 0 0 0]
 [0 0 0 0]
 [0 1 1 1]
 [1 1 1 1]
 [1 1 1 1]]


In [26]:
### Keep only safety-critical states 

In [29]:
safety_critical_flags = (safe_state_action_pairs.sum(axis=1) > 0) & (safe_state_action_pairs.sum(axis=1) < safe_state_action_pairs.shape[1])

In [30]:
safety_critical_flags

array([False,  True, False,  True,  True, False,  True, False,  True,
        True,  True, False, False,  True, False, False])

In [ ]:
env = make_frozenlake_env(
    env_map=env_map,
    task_num=0,
    is_slippery=is_slippery,
)

In [32]:
env.observation_space

Box(0.0, [ 1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1.  1. inf], (17,), float32)

In [34]:
import numpy
safety_critical_states_actions = []
for idx, flag in enumerate(safety_critical_flags):
    if flag:
        cur_state_representation = numpy.zeros(env.observation_space.shape[0])
        cur_state_representation[idx] = 1
        cur_safe_actions_multi_hot = safe_state_action_pairs[idx]
        safety_critical_states_actions.append((cur_state_representation, cur_safe_actions_multi_hot))

In [38]:
from torch.utils.data import TensorDataset

safe_actions_by_state = np.asarray(safe_state_action_pairs, dtype=np.float32)
n_states, n_actions = safe_actions_by_state.shape

safety_critical_state_mask = (
    (safe_actions_by_state.sum(axis=1) > 0)
    & (safe_actions_by_state.sum(axis=1) < n_actions)
)

safety_critical_state_indices = np.flatnonzero(safety_critical_state_mask)
safety_critical_states_actions = safe_actions_by_state[safety_critical_state_indices]

# Match OneHotWrapper's observation format: one-hot state plus final task flag.
obs_dim = n_states + 1
obs = np.zeros((len(safety_critical_state_indices), obs_dim), dtype=np.float32)
obs[np.arange(len(safety_critical_state_indices)), safety_critical_state_indices] = 1.0
obs[:, -1] = 0.0  # task flag for Task 0

labels = safety_critical_states_actions.astype(np.float32)

rashomon_dataset = TensorDataset(
    torch.tensor(obs, dtype=torch.float32),
    torch.tensor(labels, dtype=torch.float32),
)

print(f"Rashomon states: {len(rashomon_dataset)}")
print("X shape:", rashomon_dataset.tensors[0].shape)
print("Y shape:", rashomon_dataset.tensors[1].shape)
print("State indices:", safety_critical_state_indices.tolist())

Rashomon states: 8
X shape: torch.Size([8, 17])
Y shape: torch.Size([8, 4])
State indices: [1, 3, 4, 6, 8, 9, 10, 13]
